# Milestone 4 - quantitative evaluation

Scores the system with numbers only (no figures). **Runtime -> T4 GPU.** About 30-40 min the first time; re-running skips images that already exist in Drive.

**Two experiments**

1. **Ablation** - 10 epic scenes x 2 seeds, three setups on identical prompts and seeds:
   - `base` - epiCRealism with the LoRA **disabled**
   - `lora` - the Ravi Varma LoRA, short prompt (`rvrma, <scene>`)
   - `lora+expand` - LoRA + the Milestone 3 prompt expansion
2. **Pose control** - 4 held-out paintings used as OpenPose references for a *different* subject, with and without ControlNet:
   - `lora+expand` vs `lora+expand+pose`

**Metrics** (all deterministic given the images)

| metric | what it measures | how |
|---|---|---|
| **CLIP-T** | match to the user's text | CLIPScore: 100 x cosine(image, scene sentence), CLIP ViT-B/32 - a different CLIP from SD's own text encoder, so the generator is not graded by itself |
| **Style** | closeness to Ravi Varma | mean cosine similarity of CLIP ViT-L/14 image embeddings to the 20 **held-out** paintings (never trained on); for pose runs the reference painting is left out |
| **Pose IoU** | does the figure follow the reference pose | OpenPose skeleton of the output vs the reference, dilated masks, intersection-over-union |

A reference row scores the 100 real **training** paintings against the holdout - the style score a genuine Varma gets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf /content/nrv && git clone -q https://github.com/ria842/next-ravi-varma.git /content/nrv
%cd /content/nrv
!pip uninstall -y -q torchao
!pip install -q -r notebooks/requirements-gpu.txt

Optional: Gemini/Groq key from Colab Secrets for LLM expansion. Without one the template expander is used; the source is recorded per prompt either way.

In [ ]:
import os
from google.colab import userdata
for k in ('GEMINI_API_KEY', 'GROQ_API_KEY'):
    try:
        os.environ[k] = userdata.get(k); print(k, 'loaded')
    except Exception as e:
        print(k, '-', type(e).__name__)

## Config
Finds the LoRA, the holdout paintings and the training split in Drive. Upload `data/holdout` (20 images) to `My Drive/ravi-varma/holdout/` if it is not there yet.

In [ ]:
import glob, os, sys, json
sys.path[:0] = ['/content/nrv/src', '/content/nrv/scripts']

MY = '/content/drive/MyDrive'
def locate(name, likely):
    for p in likely:
        if os.path.exists(f'{MY}/{p}'):
            return f'{MY}/{p}'
    print(f'not in the usual places, searching Drive for {name} ...')
    hits = glob.glob(f'/content/drive/**/{name}', recursive=True)
    if not hits:
        raise SystemExit(f'{name} not found in Drive -- upload it, then re-run this cell')
    return hits[0]

CKPT = 'r32-750-epicrealism/checkpoint-000500'
LORA = os.path.dirname(locate(f'{CKPT}/pytorch_lora_weights.safetensors',
                              [f'ravi-varma/runs/{CKPT}/pytorch_lora_weights.safetensors']))
SARA = 'Goddess_Saraswati_by_Raja_Ravi_Varma_1896.jpg'
HOLDOUT = os.path.dirname(locate(SARA, [f'ravi-varma/holdout/{SARA}', f'ravi-varma/data/holdout/{SARA}']))
TRAIN = os.path.dirname(locate('train/metadata.jsonl', ['ravi-varma/train/metadata.jsonl',
                                                        'ravi-varma/data/train/metadata.jsonl']))
BASE  = 'emilianJR/epiCRealism'
OUT   = f'{MY}/ravi-varma/eval'
SEEDS = [42, 7]
HIRES = 1.0   # the panel ships 1.5; scores are computed at 224 px, so 1.0 halves the run time

imgs = lambda d: sorted(f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
print('LoRA   ', LORA)
print('holdout', HOLDOUT, len(imgs(HOLDOUT)), 'images')
print('train  ', TRAIN, len(imgs(TRAIN)), 'images')
assert len(imgs(HOLDOUT)) == 20, 'expected the 20 held-out paintings'

## Prompts
Fixed before any image is generated. The pose set pairs each held-out painting with a *different* subject, so ControlNet has to transfer the pose, not copy the painting.

In [ ]:
SCENES = [
    'Yudhishthira weeping over the fallen body of Karna on the battlefield of Kurukshetra',
    'Draupadi standing in the royal court, pleading with the elders',
    'Sita sitting alone in the Ashoka grove, guarded by demonesses',
    'Damayanti talking to a golden swan beside a palace pond',
    'Arjuna drawing his bow on a chariot, Krishna holding the reins',
    'Shakuntala seated under a tree in a forest hermitage, writing a letter',
    'Rama, Sita and Lakshmana walking through the forest in exile',
    'Hanuman kneeling before Rama and offering a ring',
    'Bhishma lying on a bed of arrows on the battlefield',
    'Kunti praying beside a river at dawn',
]
POSE_SET = [  # (held-out reference painting, new subject)
    ('Goddess_Saraswati_by_Raja_Ravi_Varma_1896.jpg',
     'Goddess Lakshmi seated on a rock beside a river, holding a lotus'),
    ('Raja_Ravi_Varma_Kadambari.jpg',
     'Damayanti seated on a stone bench in a palace garden, playing a veena'),
    ('Malabar_Beauty_by_Raja_Ravi_Varma.jpg',
     'Shakuntala, half-length, hand at her chest, in a forest hermitage'),
    ('Raja_Ravi_Varma_Nobleman_from_Central_India.jpg',
     'Arjuna standing in royal attire in a palace hall'),
]
for f, _ in POSE_SET:
    assert os.path.exists(f'{HOLDOUT}/{f}'), f

## Expand once, cache
Expansions are frozen in `eval/prompts.json` so a re-run (or an LLM that answers differently) cannot change the prompts being scored. Delete the file to re-expand.

In [ ]:
from expand import expand, strip_style, TRIGGER, NEGATIVE
os.makedirs(OUT, exist_ok=True)
PFILE = f'{OUT}/prompts.json'
if os.path.exists(PFILE):
    P = json.load(open(PFILE))
else:
    P = {}
    for s in SCENES + [s for _, s in POSE_SET]:
        e = expand(s, use_llm=True)
        P[s] = {'short': f'{TRIGGER}, {strip_style(s)[0]}', 'expanded': e.prompt,
                'source': e.source, 'tokens': e.tokens}
    json.dump(P, open(PFILE, 'w'), indent=1)
for s, v in P.items():
    print(f"[{v['source']}, {v['tokens']} tok] {v['expanded']}")

## Generate

In [ ]:
from PIL import Image
from generate import Painter
painter = Painter(LORA, BASE)

def path(exp, setup, i, seed):
    return f'{OUT}/images/{exp}/{setup}/{i:02d}_s{seed}.png'

jobs = []  # base first, so the LoRA is switched off once and back on once
for setup, key in [('base', 'short'), ('lora', 'short'), ('lora+expand', 'expanded')]:
    for i, s in enumerate(SCENES):
        for seed in SEEDS:
            jobs.append(('ablation', setup, i, s, seed, P[s][key], None))
for setup in ['lora+expand', 'lora+expand+pose']:
    for i, (ref, s) in enumerate(POSE_SET):
        for seed in SEEDS:
            jobs.append(('pose', setup, i, s, seed, P[s]['expanded'],
                         ref if setup.endswith('pose') else None))

todo = [j for j in jobs if not os.path.exists(path(*j[:3], j[4]))]
print(f'{len(jobs)} images, {len(todo)} to generate')
lora_on = True
for n, (exp, setup, i, s, seed, prompt, ref) in enumerate(todo, 1):
    want = setup != 'base'
    if want != lora_on:
        (painter.cn_pipe.enable_lora if want else painter.cn_pipe.disable_lora)()
        lora_on = want
    img, ctrl = painter.paint(prompt, NEGATIVE,
                              Image.open(f'{HOLDOUT}/{ref}') if ref else None, 'openpose',
                              control_scale=1.0, lora_scale=0.8, seed=seed, hires=HIRES)
    p = path(exp, setup, i, seed)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    img.save(p)
    if ctrl is not None:
        ctrl.save(f'{OUT}/images/pose/refmap_{i:02d}.png')
    print(f'{n}/{len(todo)}', exp, setup, i, seed)
if not lora_on:
    painter.cn_pipe.enable_lora()

## Score

In [ ]:
import numpy as np, pandas as pd, torch, cv2
from transformers import CLIPModel, CLIPProcessor
dev = 'cuda'

def load(name):
    return CLIPModel.from_pretrained(name, torch_dtype=torch.float16).to(dev).eval(), CLIPProcessor.from_pretrained(name)
clip_t, proc_t = load('openai/clip-vit-base-patch32')    # text alignment
clip_s, proc_s = load('openai/clip-vit-large-patch14')   # style

@torch.no_grad()
def img_emb(model, proc, ims, bs=16):
    out = []
    for k in range(0, len(ims), bs):
        x = proc(images=[im.convert('RGB') for im in ims[k:k+bs]], return_tensors='pt').to(dev)
        # vision tower + projection directly: get_image_features changed return type across transformers versions
        e = model.visual_projection(model.vision_model(pixel_values=x.pixel_values.half()).pooler_output)
        out.append(torch.nn.functional.normalize(e.float(), dim=-1))
    return torch.cat(out)

@torch.no_grad()
def txt_emb(model, proc, texts):
    x = proc(text=texts, return_tensors='pt', padding=True, truncation=True).to(dev)
    e = model.text_projection(model.text_model(input_ids=x.input_ids, attention_mask=x.attention_mask).pooler_output)
    return torch.nn.functional.normalize(e.float(), dim=-1)

hold_files = imgs(HOLDOUT)
H = img_emb(clip_s, proc_s, [Image.open(f'{HOLDOUT}/{f}') for f in hold_files])

def style(e, exclude=None):
    keep = [k for k, f in enumerate(hold_files) if f != exclude]
    return (e @ H[keep].T).mean().item()

# OpenPose agreement: dilated skeleton masks, IoU
def pose_mask(im):
    a = np.asarray(im.convert('L').resize((512, 704))) > 20
    return cv2.dilate(a.astype(np.uint8), np.ones((15, 15), np.uint8)) > 0
def pose_iou(out_img, ref_map):
    a, b = pose_mask(painter.control_image(out_img, 'openpose', out_img.size)), pose_mask(ref_map)
    if not b.any():
        return np.nan               # nothing detected in the reference
    return (a & b).sum() / (a | b).sum()

rows = []
for exp, setup, i, s, seed, prompt, ref in jobs:
    im = Image.open(path(exp, setup, i, seed))
    ref_file = POSE_SET[i][0] if exp == 'pose' else None
    e_s = img_emb(clip_s, proc_s, [im])
    cos = (img_emb(clip_t, proc_t, [im]) @ txt_emb(clip_t, proc_t, [s]).T).item()
    r = dict(experiment=exp, setup=setup, scene=i, seed=seed,
             clip_t=100 * max(cos, 0), style=style(e_s, exclude=ref_file))
    if exp == 'pose':
        r['pose_iou'] = pose_iou(im, Image.open(f'{OUT}/images/pose/refmap_{i:02d}.png'))
    rows.append(r)
df = pd.DataFrame(rows)
df.to_csv(f'{OUT}/scores.csv', index=False)

# reference: real Varma training paintings vs the holdout
T = img_emb(clip_s, proc_s, [Image.open(f'{TRAIN}/{f}') for f in imgs(TRAIN)])
ref_style = (T @ H.T).mean(dim=1).cpu().numpy()
print('scored', len(df), 'images; scores.csv written')

## Results
Mean ± std over scenes x seeds. Copy the LaTeX block into the report.

In [ ]:
def fmt(x):
    x = x.dropna()
    return f'{x.mean():.2f} ± {x.std():.2f}' if len(x) else '—'

order = [('ablation', 'base'), ('ablation', 'lora'), ('ablation', 'lora+expand'),
         ('pose', 'lora+expand'), ('pose', 'lora+expand+pose')]
table = []
for exp, setup in order:
    g = df[(df.experiment == exp) & (df.setup == setup)]
    table.append({'experiment': exp, 'setup': setup, 'n': len(g),
                  'CLIP-T': fmt(g['clip_t']), 'Style': fmt(g['style']),   # g.style is pandas' Styler
                  'Pose IoU': fmt(g['pose_iou']) if 'pose_iou' in g else '—'})
table.append({'experiment': 'reference', 'setup': 'real Varma (train set)', 'n': len(ref_style),
              'CLIP-T': '—', 'Style': f'{ref_style.mean():.2f} ± {ref_style.std():.2f}', 'Pose IoU': '—'})
summary = pd.DataFrame(table)
summary.to_csv(f'{OUT}/summary.csv', index=False)
display(summary)

# paired differences (same scene, same seed): the cleanest read of each component
def paired(exp, a, b, col):
    x = df[(df.experiment == exp) & (df.setup == a)].set_index(['scene', 'seed'])[col]
    y = df[(df.experiment == exp) & (df.setup == b)].set_index(['scene', 'seed'])[col]
    d = (y - x).dropna()
    return f'{b} - {a}: {col} {d.mean():+.2f} (better in {(d > 0).sum()}/{len(d)} pairs)'
for args in [('ablation', 'base', 'lora', 'style'), ('ablation', 'base', 'lora', 'clip_t'),
             ('ablation', 'lora', 'lora+expand', 'clip_t'), ('ablation', 'lora', 'lora+expand', 'style'),
             ('pose', 'lora+expand', 'lora+expand+pose', 'pose_iou'),
             ('pose', 'lora+expand', 'lora+expand+pose', 'clip_t'),
             ('pose', 'lora+expand', 'lora+expand+pose', 'style')]:
    print(paired(*args))

srcs = sorted({v['source'] for v in P.values()})
print('\nexpansion source:', ', '.join(srcs), '| seeds', SEEDS, '| hires', HIRES)

esc = lambda s: s.replace('±', r'$\pm$').replace('—', '--')
lines = [r'\begin{tabular}{@{}llccc@{}}', r'\toprule',
         r'Experiment & Setup & CLIP-T $\uparrow$ & Style $\uparrow$ & Pose IoU $\uparrow$ \\', r'\midrule']
for r in table:
    lines.append(f"{r['experiment']} & {r['setup']} & {esc(r['CLIP-T'])} & {esc(r['Style'])} & {esc(r['Pose IoU'])} \\\\")
lines += [r'\bottomrule', r'\end{tabular}']
print('\n' + '\n'.join(lines))